# Step 0: Remove Crosref from the Sources
- Address the error in the Crossref API retrieval method that resulted in the misclassification of retraction notices as retracted articles.
- Remove Crossref as a data source due error in Crossref API retrieval method.

Input:
- "unionlist_completed_2024-07-09.csv" renamed as "unionlist_completed_old_2024-07-09.csv"

Output:
- "unionlist_completed_2024-07-09.csv"

In [ ]:
import os
import pandas as pd

In [ ]:
"""
Read union list: 'unionlist_completed_old_2024-07-09.csv'
"""
unionlist_new = pd.read_csv("unionlist_completed_old_2024-07-09.csv").drop('Unnamed: 0',axis=1)
unionlist_new

In [ ]:
"""
Remove 'Crossref' sources from the 'source' column.
"""
## Remove exclusively 'Crossref' indexed items
unionlist_new = unionlist_new[~(unionlist_new["source"]=="Crossref")].copy()

## Remove 'Crossref' from the 'source' column for items that have multiple sources
unionlist_new["source"] = unionlist_new["source"].apply(
    lambda x: "; ".join(
        source.strip()
        for source in str(x).split(";")
        if source.strip().lower() != "crossref"
    ) if pd.notna(x) else x
)


In [ ]:
## Double-check that there are no remaining 'Crossref' sources in the 'source' column
unionlist_new[unionlist_new["source"].str.contains("Crossref", case=False, na=False)]

In [ ]:
unionlist_new

In [ ]:
## Filter and update 'coverednotindexed' files
coverednotindexed_old_dir = "coverednotindexed_old"
coverednotindexed_dir = "data/coverednotindexed"
for filename in os.listdir(coverednotindexed_old_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(coverednotindexed_old_dir, filename)
        df = pd.read_csv(filepath)
        print(f"Size of {filename} before filtering: {len(df)}")
        df = df[df["DOI"].isin(unionlist_new["DOI"])]
        df["source"] = df["source"].apply(lambda x: x.replace("Crossref;", ""))
        df["source"] = df["source"].apply(lambda x: x.replace("; Crossref", ""))
        print(f"Size of {filename} after filtering: {len(df)}")
        filepath = os.path.join(coverednotindexed_dir, filename)
        df.to_csv(filepath, index=False)

In [ ]:
getdate = {'retractionwatch': '2024-07-03',
            'pubmed': '2024-07-03',    
            'bci': '2024-07-03',
            'bioabs': '2024-07-03',
            'ccc': '2024-07-03',
            'webofsciencecore': '2024-07-03',
            'crossref': '2024-07-03'
          }
data_dir = 'retractionyear_old/'
pubmed= pd.read_csv(data_dir+'pubmed_retractionyear_'+getdate['pubmed']+'.csv')
bci = pd.read_csv(data_dir+'bci_retractionyear_'+getdate['bci']+'.csv')
ccc = pd.read_csv(data_dir+'ccc_retractionyear_'+getdate['ccc']+'.csv')
bioabs= pd.read_csv(data_dir+'bioabs_retractionyear_'+getdate['bioabs']+'.csv')
woscore = pd.read_csv(data_dir+'webofsciencecore_retractionyear_'+getdate['webofsciencecore']+'.csv')
retractionwatch = pd.read_csv(data_dir+'retractionwatch_'+getdate['retractionwatch']+'.csv',\
                             encoding='latin1').rename(
    columns={'OriginalPaperDOI':'DOI', 
             'OriginalPaperPubMedID': 'PubMedID', 
             'OriginalPaperDate': 'Year'})

retractionwatch['PubMedID']= retractionwatch['PubMedID'].fillna(0).astype(int)\
                .replace(0,'').astype(str).str.strip()

retractionwatch['RetractionPubMedID']= retractionwatch['RetractionPubMedID'].fillna(0).astype(int)\
                .replace(0,'').astype(str)

retractionwatch['Year']=  pd.to_datetime(retractionwatch['Year'], exact=False).dt.year
retractionwatch['RetractionYear']=  pd.to_datetime(retractionwatch['RetractionDate'], exact=False).dt.strftime("%Y").fillna(0).astype(int)

In [ ]:
sources = [pubmed, bci, ccc, bioabs, woscore, retractionwatch]
dois = set()
for source in sources:
    source = source[source["RetractionYear"] != 0]
    dois.update(set(source["DOI"]))
dois = list(dois)
dois = [doi for doi in dois if str(doi) != 'nan']
dois = set(dois)

In [ ]:
len(dois)

In [ ]:
## Read Retraction Year from Crossref
crossref = pd.read_csv(data_dir+'crossref_retractionyear_'+getdate['crossref']+'.csv')
crossref = crossref[crossref["RetractionYear"] != 0]
dois_crossref = set(crossref["DOI"])

In [ ]:
unionlist_new.value_counts("RetractionYear")

In [ ]:
count_items_with_retraction_year = unionlist_new[unionlist_new["RetractionYear"] != 0]
len(count_items_with_retraction_year)

In [ ]:
## Update RetractionYear for items that are in Crossref
for i, row in unionlist_new.iterrows():
    if row["DOI"] in dois_crossref and row["DOI"] not in dois:
        unionlist_new.at[i, "RetractionYear"] = 0
unionlist_new.value_counts("RetractionYear")

In [ ]:
## Save the updated unionlist CSV file
unionlist_new.to_csv("data/unionlist/unionlist_completed_2024-07-09.csv", index=False)